# 00 — Project Setup and Data Validation

Run this notebook first to confirm the repo can find its folders and that raw data has been placed in the expected structure.

In [ ]:
from pathlib import Path
import os
import json
import sys

def find_project_root(start=None):
    if "TCGA_TAA_PROJECT_ROOT" in os.environ:
        return Path(os.environ["TCGA_TAA_PROJECT_ROOT"]).expanduser().resolve()
    if start is None:
        start = Path.cwd()
    start = Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "config" / "genes.json").exists():
            return candidate
    raise FileNotFoundError("Could not find config/genes.json. Start Jupyter from repo root.")

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

with open(PROJECT_ROOT / "config" / "genes.json") as f:
    genes = json.load(f)

print("TAA targets:", genes["taa_target_genes"])

In [ ]:
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"

expected = {
    "RNA files": RAW / "rna_expression" / "files",
    "RNA download info": RAW / "rna_expression" / "download_info",
    "Copy-number files": RAW / "copy_number" / "files",
    "Copy-number download info": RAW / "copy_number" / "download_info",
    "Mutation files": RAW / "mutation" / "files",
    "Mutation download info": RAW / "mutation" / "download_info",
    "Clinical": RAW / "clinical",
    "Processed": PROCESSED,
    "Results": RESULTS,
}

for label, path in expected.items():
    print(f"{label:28s} | exists={path.exists()} | {path}")

In [ ]:
def count_files(folder, pattern):
    if not folder.exists():
        return 0
    return sum(1 for _ in folder.rglob(pattern))

checks = {
    "RNA TSV files": (RAW / "rna_expression" / "files", "*.tsv"),
    "Copy-number TSV files": (RAW / "copy_number" / "files", "*.tsv"),
    "Mutation MAF.GZ files": (RAW / "mutation" / "files", "*.maf.gz"),
    "Partial files": (RAW, "*.partial"),
    "Clinical TSV files": (RAW / "clinical", "*.tsv"),
}

for label, (folder, pattern) in checks.items():
    print(f"{label:24s}: {count_files(folder, pattern)}")

If the counts look reasonable and partial files are `0`, move to `01_Data_Processing_Pipeline.ipynb`.